# Salient Object Detection demo

**Setup:** Train first, e.g. `python train.py --data_root data/synthetic --images images --masks masks --run_name demo`. Set `CKPT_PATH` below.

In [ ]:
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F

from checkpoint_io import torch_load_ckpt
from sod_model import build_model

CKPT_PATH = "checkpoints/demo/best.pt"
VARIANT = "baseline"
IMAGE_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def infer(image_path: str):
    pil = Image.open(image_path).convert("RGB")
    w, h = pil.size
    arr = np.asarray(pil, dtype=np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = F.interpolate(
        x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )

    model = build_model(VARIANT, in_ch=3, base=32).to(device)
    ckpt = torch_load_ckpt(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    x = x.to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    infer_ms = (time.perf_counter() - t0) * 1000.0

    mask = F.interpolate(pred, size=(h, w), mode="bilinear", align_corners=False)
    mask = mask[0, 0].cpu().numpy()

    heat = plt.cm.hot(mask)[..., :3]
    overlay = np.clip(
        arr * (1 - 0.45 * mask[..., None]) + heat * (0.45 * mask[..., None]),
        0,
        1,
    )

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(arr)
    axes[0].set_title("Input")
    axes[1].imshow(mask, cmap="inferno")
    axes[1].set_title("Predicted mask")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[3].imshow(arr)
    axes[3].contour(mask, levels=[0.5], colors="cyan")
    axes[3].set_title("Contour")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(
        f"Inference: {infer_ms:.2f} ms ({IMAGE_SIZE}px forward; original {w}x{h})"
    )
    plt.tight_layout()
    plt.show()
    return infer_ms


# infer("your_image.jpg")
Path(CKPT_PATH).exists()